# LLM Augmented Query: RAG with Oracle Vector Search & OpenAI

**Copyright 2026, Denis Rothman**

## Overview
This notebook implements a **Retrieval-Augmented Generation (RAG)** pipeline. It bridges the gap between enterprise data stored in an **Oracle Database** and Generative AI. By leveraging Oracle's native Vector Search capabilities, we retrieve precise, context-aware information to ground the responses generated by an OpenAI Large Language Model (LLM).

## Key Technologies
* **Database:** Oracle Database (utilizing the `VECTOR` data type and `VECTOR_DISTANCE` functions).
* **LLM & Embeddings:** OpenAI (`text-embedding-3-small` for vectorization, `gpt-5.2` for generation).
* **Driver:** `oracledb` (Thick/Thin mode for Python).

## Workflow Steps

*   **Setup & Connection:** Establish a secure connection to the Oracle Database using Wallet credentials.

*   **Vector Store Verification:** Inspect the `CONTEXT_LIBRARY` and `KNOWLEDGE_STORE` tables to ensure vector columns are ready.

*   **Embedding Generation:** Convert user queries into vector embeddings using OpenAI.

*   **Semantic Search:** Perform similarity searches in Oracle to retrieve relevant context and knowledge snippets.

*   **Augmented Generation:** Construct a prompt combining the user query with retrieved database content and generate a grounded response.

# 1.Installation and Setup

## Setting Oracle up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
wallet_path = "/content/drive/MyDrive/oracle_wallet"

In [ ]:
# 1.Installation and Setup
# -------------------------------------------------------------------------
# We install specific versions for stability and reproducibility.
# We include tiktoken for token-based chunking and tenacity for robust API calls.

In [ ]:
!pip install oracledb==3.4.1

In [ ]:
import oracledb
print(oracledb.__version__)

## Oracle connection

In [ ]:
from google.colab import userdata
# Pulling the Oracle DSN from the hidden Colab Secrets (Safe for GitHub!)
oracle_dsn = userdata.get('oracle_dsn')

In [ ]:
import os

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", oracle_dsn)
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

##  Checking the Oracle tables

In [ ]:
# Verify that the tables exist and the VECTOR columns are correct

print("Checking Oracle vector tables...\n")

cursor.execute("""
SELECT table_name
FROM user_tables
WHERE table_name IN ('CONTEXT_LIBRARY', 'KNOWLEDGE_STORE')
""")

tables = cursor.fetchall()
print("Tables found:", tables)

print("\nColumn definitions for CONTEXT_LIBRARY:")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'CONTEXT_LIBRARY'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)

print("\nColumn definitions for KNOWLEDGE_STORE:")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'KNOWLEDGE_STORE'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)

print("\nRunning a simple test query on DUAL...")
cursor.execute("SELECT 'Oracle connection OK' FROM dual")
print(cursor.fetchone()[0])


## Installing OpenAI

In [ ]:
!pip install tqdm==4.67.1 --upgrade
!pip install openai==2.14.0

In [ ]:
# Imports and API Key Setup
# We will use the OpenAI library to interact with the LLM and Google Colab's
# secret manager to securely access your API key.

import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    api_key = userdata.get("API_KEY")
    if not api_key:
        raise userdata.SecretNotFoundError("API_KEY not found.")

    # Set environment variable for downstream tools/libraries
    os.environ["OPENAI_API_KEY"] = api_key

    # Create client (will read from OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI API key loaded and environment variable set successfully.")

except userdata.SecretNotFoundError:
    print('Secret "API_KEY" not found.')
    print('Please add your OpenAI API key to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the API key: {e}")

# Configuration
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 # Dimension for text-embedding-3-small
GENERATION_MODEL = "gpt-5.2"

## Miscellaneous

In [ ]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken
from tenacity import retry, stop_after_attempt, wait_random_exponential
# general imports required in the notebooks of this book
import re
import textwrap
from IPython.display import display, Markdown
import copy

# 2.Helper Functions for Chunking and Embedding

In [ ]:
# Helper Functions for Chunking and Embedding
# -------------------------------------------------------------------------

# Initialize tokenizer for robust, token-aware chunking
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=400, overlap=50):
    """Chunks text based on token count with overlap (Best practice for RAG)."""
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = tokenizer.decode(chunk_tokens)
        # Basic cleanup
        chunk_text = chunk_text.replace("\n", " ").strip()
        if chunk_text:
            chunks.append(chunk_text)
    return chunks

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    """Generates embeddings for a batch of texts using OpenAI, with retries."""
    # OpenAI expects the input texts to have newlines replaced by spaces
    texts = [t.replace("\n", " ") for t in texts]
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]


# 3.LLM Augmented Query

### 3. Vector System Verification

In [ ]:
import oracledb

# Prepare a test query
test_query = "What is the purpose of this system?"
query_embedding = get_embeddings_batch([test_query])[0]

cursor.setinputsizes(query_vec=oracledb.DB_TYPE_VECTOR)

cursor.execute("""
    SELECT id, source, text,
           VECTOR_DISTANCE(embedding, :query_vec) AS distance
    FROM knowledge_store
    ORDER BY embedding <-> :query_vec
    FETCH FIRST 5 ROWS ONLY
""",
{
    "query_vec": query_embedding
})

results = cursor.fetchall()

print("\nTop 5 vector search results:\n")
for r in results:
    text_clob = r[2]              # this is a CLOB
    text_str = text_clob.read()   # convert to Python string

    print(f"ID: {r[0]}")
    print(f"Source: {r[1]}")
    print(f"Text snippet: {text_str[:120]}...")
    print(f"Distance: {r[3]}")
    print("-" * 60)

## 4.Defining the Oracle Vector Search Function

The core of our retrieval pipeline is the `oracle_vector_search` function. This utility performs a semantic similarity search against our specific tables (`KNOWLEDGE_STORE` or `CONTEXT_LIBRARY`).

**Key Technical Implementations:**
1.  **Vector Type Mapping:** We explicitly map the input Python list to Oracle's native vector type using `cursor.setinputsizes(query_vec=oracledb.DB_TYPE_VECTOR)`. This ensures the driver correctly serializes the data for the database engine.
2.  **Distance Calculation:** The SQL query uses the `<->` operator (Euclidean distance) in the `ORDER BY` clause to efficiently sort records by finding the "nearest neighbors" to the query vector.
3.  **Scoring:** We also select `VECTOR_DISTANCE` to return the precise similarity score, allowing us to gauge how relevant a retrieved snippet is to the user's prompt.
4.  **Dynamic Retrieval:** The function adapts the `SELECT` statement based on the target table, ensuring we retrieve the correct text fields (`description` for blueprints vs. `text` for raw knowledge).

In [ ]:
import oracledb

def oracle_vector_search(table, embedding, top_k=5):
    cursor.setinputsizes(query_vec=oracledb.DB_TYPE_VECTOR)

    if table.lower() == "knowledge_store":
        sql = f"""
            SELECT id, source, text,
                   VECTOR_DISTANCE(embedding, :query_vec) AS distance
            FROM {table}
            ORDER BY embedding <-> :query_vec
            FETCH FIRST {top_k} ROWS ONLY
        """
        cursor.execute(sql, {"query_vec": embedding})
        # rows: (id, source, text_clob, distance)
        return cursor.fetchall()

    elif table.lower() == "context_library":
        sql = f"""
            SELECT id, description,
                   VECTOR_DISTANCE(embedding, :query_vec) AS distance
            FROM {table}
            ORDER BY embedding <-> :query_vec
            FETCH FIRST {top_k} ROWS ONLY
        """
        cursor.execute(sql, {"query_vec": embedding})
        # rows: (id, description_clob, distance)
        return cursor.fetchall()

    else:
        raise ValueError(f"Unsupported table: {table}")


## 4.1 Handling Oracle LOB Data

When retrieving large text fields (like descriptions or document content) from Oracle Database, the `oracledb` driver often returns them as **LOB (Large Object)** objects rather than standard Python strings.

The helper function `read_clob_or_str` is designed to handle this data type variability robustly:
1.  **LOB Detection:** It checks if the returned value has a `.read()` method, which is characteristic of Oracle `CLOB` objects in the Python driver.
2.  **Conversion:** If it is a LOB, it reads the stream and converts it to a string.
3.  **Safety:** It includes safe fallbacks for `None` values or standard strings, preventing the pipeline from crashing if a field is empty or already in string format.

In [ ]:
def read_clob_or_str(value):
    """Return string content for CLOB-like objects or plain strings; safe for None."""
    if value is None:
        return ""
    if hasattr(value, "read"):
        try:
            return value.read() or ""
        except Exception:
            return ""
    return str(value)

## 4.2. Robust Text Extraction from Database Rows

Since we are querying two different tables (`CONTEXT_LIBRARY` and `KNOWLEDGE_STORE`) with potentially different column structures, relying on hard-coded indices can be risky.

The `extract_text_from_row` function adds a layer of flexibility and fault tolerance to our retrieval process:
* **Priority Extraction:** It attempts to read from a `preferred_index` if you know exactly where the text lives (e.g., column 2).
* **Heuristic Fallback:** If the preferred column is missing or empty, it automatically checks common text positions (index 2, then index 1).
* **Safety Net:** As a last resort, it concatenates all string-like data in the row, ensuring that *some* content is always returned to the LLM rather than crashing the pipeline.

This function relies on `read_clob_or_str` to ensure that whether the data is a simple string or a complex Oracle LOB, it is correctly converted to text.

In [ ]:
def extract_text_from_row(row, preferred_index=None):
    """
    Extract the most likely text field from a DB row.
    - If preferred_index is provided and valid, use it.
    - Otherwise try common positions (2 then 1) and fall back to joining string-like columns.
    """
    if row is None:
        return ""
    if preferred_index is not None and preferred_index < len(row):
        return read_clob_or_str(row[preferred_index])

    if len(row) >= 3:
        candidate = read_clob_or_str(row[2])
        if candidate:
            return candidate
    if len(row) >= 2:
        return read_clob_or_str(row[1])
    return " ".join(read_clob_or_str(c) for c in row if read_clob_or_str(c))

## 5.Orchestrating the RAG Pipeline

This final cell brings all the previous components together into a cohesive, executable workflow. It is designed to be **defensive**, meaning it checks for existing variables (like `query_embedding`) before re-computing them, saving time and API costs during iterative development.

The workflow proceeds in four distinct stages:
1.  **Retrieval & Pre-processing:** It calls `oracle_vector_search` to fetch relevant rows, then uses `extract_text_from_row` and `trim_snippet` to clean and truncate the text, ensuring it fits within the LLM's context window.
2.  **Prompt Assembly:** It combines the system instructions, the retrieved "Knowledge Base" (facts), the "Context Library" (blueprints/styles), and the user's query into a single, structured prompt.
3.  **Generation:** It sends this augmented prompt to the `gpt-5.2` model to generate the final response.
4.  **Reporting:** It generates a formatted Markdown report that displays the retrieved evidence alongside the final answer, plus a breakdown of execution times for performance tuning.

In [ ]:
# --- Timed retrieval + generation with pretty Markdown output ---
import time
from datetime import timedelta
import textwrap
from IPython.display import display, Markdown

# Ensure system_prompt exists
system_prompt = globals().get("system_prompt", "You are a strategic assistant helping a product team communicate clearly and effectively.")

# Start overall timer
overall_start = time.perf_counter()

# --- Retrieval timer: embeddings + DB searches ---
retrieval_start = time.perf_counter()

# Ensure we have a query embedding and search results (reuse or compute)
if 'query_embedding' not in globals():
    if 'user_query' in globals() and callable(globals().get('get_embeddings_batch', None)):
        query_embedding = get_embeddings_batch([user_query])[0]
    else:
        raise NameError("Missing 'query_embedding' and cannot compute it: ensure 'user_query' and 'get_embeddings_batch' are defined.")

if 'context_results' not in globals():
    if callable(globals().get('oracle_vector_search', None)):
        context_results = oracle_vector_search("context_library", query_embedding, top_k=3)
    else:
        context_results = []

if 'knowledge_results' not in globals():
    if callable(globals().get('oracle_vector_search', None)):
        knowledge_results = oracle_vector_search("knowledge_store", query_embedding, top_k=5)
    else:
        knowledge_results = []

# Build snippet lists with safe extraction
context_snippets = [extract_text_from_row(r, preferred_index=1) for r in context_results]
knowledge_snippets = [extract_text_from_row(r, preferred_index=2) for r in knowledge_results]

# Token-aware trimming
def tokens_of(text):
    return len(tokenizer.encode(text))

MAX_SNIPPET_TOKENS = 400
def trim_snippet(s, max_tokens=MAX_SNIPPET_TOKENS):
    if not s:
        return ""
    try:
        if tokens_of(s) <= max_tokens:
            return s.strip()
        tokens = tokenizer.encode(s)
        return tokenizer.decode(tokens[:max_tokens]).strip()
    except Exception:
        return s.strip()[:4000]

context_snippets = [trim_snippet(s) for s in context_snippets if s]
knowledge_snippets = [trim_snippet(s) for s in knowledge_snippets if s]

retrieval_end = time.perf_counter()
retrieval_elapsed = retrieval_end - retrieval_start

# --- Build prompt ---
user_query = globals().get("user_query", "Using the official product spec sheet, write a short marketing description for the new QuantumDrive Q-1. The description should be confident, aspirational, and focus on the benefits for creative professionals. Please cite your sources.")

context_block = "\n\n".join(f"- {s}" for s in context_snippets) if context_snippets else "- (no context found)"
knowledge_block = "\n\n".join(f"- {s}" for s in knowledge_snippets) if knowledge_snippets else "- (no knowledge found)"

final_prompt = f"""
Context Library:
{context_block}

Knowledge Base:
{knowledge_block}

User Query:
{user_query}

Answer:
"""

# --- Generation timer: LLM call ---
generation_start = time.perf_counter()
try:
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": final_prompt}
        ],
        temperature=0.4,
        max_completion_tokens=800
    )

    # Extract content robustly
    content = None
    if isinstance(response, dict):
        content = response.get("choices", [{}])[0].get("message", {}).get("content")
    else:
        try:
            content = response.choices[0].message.content
        except Exception:
            content = str(response)

except Exception as e:
    generation_error = e
    content = None

generation_end = time.perf_counter()
generation_elapsed = generation_end - generation_start

overall_end = time.perf_counter()
overall_elapsed = overall_end - overall_start

# --- Prepare Markdown output ---
def fmt(seconds):
    return str(timedelta(seconds=round(seconds, 3)))

def mk_snippet_section(title, snippets, max_chars=1000):
    if not snippets:
        return f"**{title}**\n\n- (none found)\n"
    lines = [f"**{title}**\n"]
    for i, s in enumerate(snippets, start=1):
        # wrap and truncate for readability
        wrapped = textwrap.fill(s, width=100)
        if len(wrapped) > max_chars:
            wrapped = wrapped[:max_chars].rstrip() + "…"
        lines.append(f"{i}. {wrapped}\n")
    return "\n".join(lines)

md_parts = []

# Header
md_parts.append("# GPT Augmented Query Results\n")

# Query
md_parts.append("### User Query\n")
md_parts.append(f"> {textwrap.fill(user_query, width=100)}\n")

# Retrieved context and knowledge
md_parts.append("### Retrieved Context\n")
md_parts.append(mk_snippet_section("Context Library snippets", context_snippets))

md_parts.append("### Retrieved Knowledge\n")
md_parts.append(mk_snippet_section("Knowledge Base snippets", knowledge_snippets))

# Model output
md_parts.append("### Model Response\n")
if content:
    # Indent the model response as a blockquote for visual clarity
    response_wrapped = "\n".join("> " + line for line in textwrap.fill(content, width=100).splitlines())
    md_parts.append(response_wrapped + "\n")
else:
    md_parts.append("_No content returned from the model._\n")
    if 'generation_error' in globals():
        md_parts.append(f"**Generation error:** `{str(generation_error)}`\n")

# Timing summary
md_parts.append("### Timing Summary\n")
md_parts.append(f"- **Retrieval time** (embeddings + DB searches): **{fmt(retrieval_elapsed)}**")
md_parts.append(f"- **Generation time** (LLM call): **{fmt(generation_elapsed)}**")
md_parts.append(f"- **Total elapsed time**: **{fmt(overall_elapsed)}**\n")

# Combine and display
md_text = "\n\n".join(md_parts)
display(Markdown(md_text))